# Experiment: Balancing Lexical and Semantic Retrieval in the Medical Domain
Welcome to the main evaluation notebook for our research!

**Our Objective:**
To empirically evaluate how different Information Retrieval (IR) models handle varying levels of query verbosity, and to find the optimal balance between exact lexical matching and semantic understanding.

**The Setup:**
* **Dataset:** TREC-COVID (CORD-19 collection).
* **Query Verbosity:** We evaluate three distinct query levels provided by the dataset: `Title` (Short), `Description` (Medium), and `Narrative` (Long).
* **Evaluation Metrics:** We primarily focus on **nDCG@10** (to ensure the most highly relevant documents appear at the top) and **MAP** (to assess overall ranking quality).

In this notebook, we will progress through three phases:
1.  **Lexical Baselines:** BM25, and RM3 (Query Expansion).
2.  **Dense Retrieval:** Standard DPR and bioDPR.
3.  **Hybrid Search:** Combining BM25 and bioDPR using Reciprocal Rank Fusion (RRF).


In [2]:
import pyterrier as pt
import pandas as pd
import os
from pathlib import Path
import warnings
import pyterrier_dr as ptdr
from dpr_dense import make_dpr_pipeline

warnings.filterwarnings('ignore', category=DeprecationWarning)
if not pt.started():
    pt.init()

## Data Preparation
We need to laod the data and create dataframes for the different query verbosity levels (title, description, narrative).

In [15]:
dataset_name = "irds:cord19/trec-covid"
dataset = pt.get_dataset(dataset_name)

In [16]:
topics_data = []
for query in dataset.irds_ref().queries_iter():
    topics_data.append({
        'qid': query.query_id,
        'title': query.title,
        'description': query.description,
        'narrative': query.narrative
    })
df_all_topics = pd.DataFrame(topics_data)

In [17]:
# TITLE Queries
topics_title = df_all_topics[['qid', 'title']].copy()
topics_title = topics_title.rename(columns={'title': 'query'})

# DESCRIPTION Queries
topics_desc = df_all_topics[['qid', 'description']].copy()
topics_desc = topics_desc.rename(columns={'description': 'query'})

# NARRATIVE Queries
topics_narr = df_all_topics[['qid', 'narrative']].copy()
topics_narr = topics_narr.rename(columns={'narrative': 'query'})

## Phase 1: Lexical Baseline & Query Expansion

We begin by loading the sparse index that will be used by both BM25 and the RM3 expansion. If you don't have the index built yet follow the instruction in `index_creation/sparse_index.ipynb` to build it first.

In [18]:
index_dir_path = Path("../index/trec_covid_index_default").resolve()
index_dir = str(index_dir_path)
index_properties_file = os.path.join(index_dir, "data.properties")

In [19]:
if os.path.exists(index_properties_file):
    print(f"Index found at {index_dir}. Loading it...")
    index_ref = pt.IndexRef.of(index_properties_file)
else:
    print(f"Index not found at {index_dir}. Make sure to build the index first using `index_creation/sparse_index.ipynb`.")

Index found at C:\Users\zosia\Desktop\Studies\Information Retrieval\IR_Project\index\trec_covid_index_default. Loading it...


In [20]:
bm25 = pt.terrier.Retriever(index_ref, wmodel="BM25")
rm3_pipe = bm25 >> pt.rewrite.RM3(index_ref) >> bm25

## Phase 2: Dense Semantic Retrieval

Now we need to extend our experiment by dense retrival models. We will load two indexes: one for a general dense retriever (DPR) and another for a biomedical domain-specific dense retriever (BioDPR). If you don't have indexes built yet follow the instruction in `index_creation/dpr_index.ipynb` and `index_creation/biodpr_index.ipynb`.


In [27]:
dense_index_dir = str(Path("../index/trec_covid_dense").resolve())

if os.path.exists(os.path.join(dense_index_dir, "pt_meta.json")):
    print("Found downloaded dense index! Loading...")
    dpr_pipe = make_dpr_pipeline(
        dataset,
        dense_index_dir,
    )
    
    print("DPR Pipeline ready!")
else:
    print(f"Index not found at {index_dir}. Make sure to build the index first using `index_creation/dpr_index.ipynb`.")

Found downloaded dense index! Loading...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DPR Pipeline ready!


In [28]:
biodpr_index_dir = str(Path("../index/trec_covid_biodpr_complete").resolve())

if os.path.exists(os.path.join(biodpr_index_dir, "pt_meta.json")):
    print("Found BioDPR index! Loading...")
    
    biodpr_model = ptdr.SBertBiEncoder("pritamdeka/S-PubMedBert-MS-MARCO")
    biodpr_index = ptdr.FlexIndex(biodpr_index_dir)
    biodpr_pipe = biodpr_model >> biodpr_index
    
    print("BioDPR Pipeline ready!")
else:
    print(f"Index not found at {index_dir}. Make sure to build the index first using `index_creation/biodpr_index_complete.ipynb`.")

Found BioDPR index! Loading...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-PubMedBert-MS-MARCO
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BioDPR Pipeline ready!


## Phase 3: Hybrid Model

Finally, we will create a hybrid model by combining the BM25 and BioDPR pipelines using Reciprocal Rank Fusion (RRF). This will allow us to leverage the strengths of both lexical and dense retrieval methods.

In [29]:
def apply_rrf(res, k=60):
    res = res.copy()
    res['score'] = 1.0 / (k + res['rank'] + 1)
    return res

In [30]:
rrf_transformer = pt.apply.generic(apply_rrf)
bm25_rrf = bm25 >> rrf_transformer
biodpr_rrf = biodpr_pipe >> rrf_transformer
hybrid_pipe = bm25_rrf + biodpr_rrf

## Phase 4: Experiment

Lastly we conduct the experiment by evaluating all the models (BM25, RM3, DPR, BioDPR, and Hybrid) across the three query verbosity levels (Title, Description, Narrative) using the MAP and nDCG@10 evaluation metrics.

In [31]:
models = [bm25, rm3_pipe, dpr_pipe, biodpr_pipe, hybrid_pipe]
model_names = ["BM25", "BM25 + RM3", "DPR", "BioDPR", "Hybrid"]
eval_metrics = ["ndcg_cut_10", "map"]
query_variants = {
    "Title": topics_title,
    "Description": topics_desc,
    "Narrative": topics_narr
}

In [32]:
for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=model_names,
        baseline=0
    )
    
    display(experiment_results)
    print("\n")
    

Results for TITLE Queries:


C:\Users\zosia\anaconda3\Lib\site-packages\pyterrier\_evaluation\_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #4: Hybrid (<pyterrier._ops.Sum object at 0x000001B91E6DA720>)
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

...

C:\Users\zosia\anaconda3\Lib\site-packages\pyterrier\_evaluation\_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #4: Hybrid (<pyterrier._ops.Sum object at 0x000001B91E6DA720>)
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,ndcg_cut_10,map +,map -,map p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.518681,NaN,NaN,NaN,NaN,NaN,NaN
1,BM25 + RM3,0.175874,0.489098,19.0,31.0,1.503040e-01,17.0,27.0,0.135629
2,DPR,0.111529,0.368355,14.0,36.0,2.405376e-03,11.0,35.0,0.000409
3,BioDPR,0.140223,0.551742,20.0,30.0,2.476971e-01,25.0,23.0,0.409129
4,Hybrid,0.207661,0.640086,41.0,9.0,1.232110e-07,34.0,13.0,0.000133


## Key Findings & Analysis
For key finding and analysis of our results please refer to our paper!

## Extensions: Task-Specific Fine-Tuning

To test whether bridging the vocabulary gap can improve the performance, we used bioDPR which is a general biomedical dense retriever. However, better results should be possible by fine-tuning the dense retriever directly on the Covid-19-related dataset. You can read more about the idea below.

As observed in our results, out-of-the-box (zero-shot) dense retrieval models often underperform strong lexical baselines like BM25 in highly specialized domains. To close this gap, dense models require **task-specific fine-tuning** using domain relevance judgments.

For future work, we have provided an experimental fine-tuning pipeline. 

### The Fine-Tuning Pipeline (`extensions/dpr_finetune.py`)
This script uses `MultipleNegativesRankingLoss` to fine-tune our bi-encoders directly on the TREC-COVID qrels. 

**How it works:**
1.  **Positive Pairs:** It matches query text with the text of known relevant documents (label >= 1).
2.  **Verbosity Impact:** The script supports fine-tuning on specific verbosity levels (e.g., training exclusively on `Title` queries vs. `Narrative` queries) to create highly specialized medical encoders.
3. **Dataset**: Remember not to use this test set for fine-tuning; only use the training set qrels to avoid data leakage and ensure a fair evaluation.

Due to the heavy computational requirements (GPU recommended), this fine-tuning step is left as a repository extension and is not executed live in this evaluation notebook.

### Snapshot: BM25 vs DPR (MiniLM) from this notebook

Values below are taken from **embedded `pt.Experiment` output** in this notebook (dense model **`sentence-transformers/all-MiniLM-L6-v2`**, same title+abstract field as BM25). Re-run the **first DPR comparison block** (BM25 … DPR all-MiniLM) after changing code or data to refresh metrics.

| Query formulation | BM25 MAP | DPR MAP | BM25 nDCG@10 | DPR nDCG@10 |
|-------------------|----------|---------|--------------|-------------|
| Title             | 0.207   | 0.075   | 0.602        | 0.299       |
| Description       | 0.222   | 0.120   | 0.623        | 0.449       |
| Narrative         | 0.158   | 0.112   | 0.519        | 0.368       |

**Readout:** lexical **BM25** is stronger on MAP in this setup; **DPR** improves with **richer queries** (description vs title) but remains below BM25 on MAP here—consistent with domain vocabulary and indexing only title+abstract. **Fine-tuning** (`dpr_finetune.py`) is the intended next step to close that gap without replacing the sparse baseline.

### Optional: training-pair scale and CLI

`dpr_finetune.build_training_pairs` needs the corpus, a topics table (`qid`, title, description, narrative), and qrels with a `label` column. A **full** run uses all judgments with `label >= 1` and can yield on the order of **many tens of thousands** of (query, document) pairs when `verbosity="all"`.

Sanity-check without the notebook, from the `src/` directory:

`python dpr_finetune.py --pairs-smoke`

Full training (slow on CPU; GPU recommended):

`python dpr_finetune.py --train --verbosity all`


### Fine-tuning outside the notebook

Full **fine-tuning and re-indexing** are intentionally **not** run here (heavy on CPU; use a **GPU** when possible). After `python dpr_finetune.py --train`, build a **FlexIndex** with the same `iter_flex_docs` + `doc_encoder >> indexer` workflow as in `dpr_dense.py`, then use **`retriever_for_loaded_encoder`** on that directory so `pt.Experiment` sees **`docno`** in rankings.
